
# CUMULUS — paper-aligned reproduction

This notebook reconstructs the CUMULUS benchmark in one place.

It implements the paper's design space of **27 complete pipelines**:

- **Imputation:** Mean, LOCF, KNN (`k=5`)
- **Outlier handling:** Z-score (`|z|>3`), MAD, Isolation Forest
- **Normalization:** Min-Max, Z-score, RobustScaler

The stage order is:

**Imputation → Outlier handling → Normalization**

Controlled missingness and spike corruption are evaluated at **5%, 10%, 15%, and 20%** using fixed random seeds.

### Important transparency rule

The literal paper values in `results/tables/paper_table2_reference.csv` and `paper_table3_reference.csv` are stored separately from recomputed outputs.


## 1. Configuration

In [ ]:

from pathlib import Path

# Local dataset paths used for the current reproduction.
D2_DIR = Path(r"C:\Users\oxije\Dropbox\Dissertation\Daten\01_Datensaetze_Original\02_Autism_EyeTracking\Eye-tracking Output")
CHEATING_DIR = Path(r"C:\Users\oxije\Dropbox\Dissertation\Daten\02_Eigenes_Experiment\Cheating_Detection_Experiment\Originaldaten_split\alle\eyetracking_Cheating")

# Repository-relative outputs.
ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"

for p in [TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# "quick": first file from each dataset + 5% only.
# "full": all files + 5/10/15/20%.
RUN_MODE = "quick"

BASE_SEED = 42
MISSING_LEVELS_FULL = [5, 10, 15, 20]
OUTLIER_LEVELS_FULL = [5, 10, 15, 20]

ZSCORE_THRESHOLD = 3.0
MAD_THRESHOLD = 3.0
IFOREST_CONTAMINATION = 0.05

USE_CHECKPOINTS = True
FORCE_RECOMPUTE = False

print("RUN_MODE:", RUN_MODE)
print("D2:", D2_DIR)
print("Cheating:", CHEATING_DIR)
print("Tables:", TABLE_DIR)
print("Figures:", FIGURE_DIR)


In [ ]:

import hashlib
import pickle
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, ks_2samp, zscore, median_abs_deviation, skew, kurtosis
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

FEATURE_COLS = ["Gaze X", "Gaze Y", "ET_PupilLeft", "ET_PupilRight"]
IMPUTERS = ["mean", "locf", "knn(k=5)"]
OUTLIER_METHODS = ["zscore", "mad", "iforest"]
SCALERS = ["minmax", "zscore", "robust"]

PIPELINES = pd.DataFrame([
    {
        "imputer": i,
        "outlier_method": o,
        "scaler": s,
        "pipeline": f"{i} -> {o} -> {s}",
    }
    for i, o, s in product(IMPUTERS, OUTLIER_METHODS, SCALERS)
])

assert len(PIPELINES) == 27
display(PIPELINES)


## 2. Paper reference tables

In [ ]:

paper_table2 = pd.read_csv(TABLE_DIR / "paper_table2_reference.csv")
paper_table3 = pd.read_csv(TABLE_DIR / "paper_table3_reference.csv")

display(paper_table2)
display(paper_table3)



## 3. Canonical eye-tracking representation

All recordings are mapped to:

`Gaze X, Gaze Y, ET_PupilLeft, ET_PupilRight`

Invalid samples are represented as `NaN`. Binocular gaze coordinates are averaged where necessary. Monocular pupil data are mapped to the left-pupil channel while the unavailable right-pupil channel remains missing.


In [ ]:

def find_data_files(root: Path):
    files = list(root.glob("*.csv")) + list(root.glob("*.tsv"))
    return sorted([p for p in files if p.is_file()])


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", ".", regex=False)
         .replace({"-": np.nan, "nan": np.nan, "None": np.nan, "": np.nan}),
        errors="coerce",
    )


def load_eye_file(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".tsv":
        df = pd.read_csv(path, sep="\t", low_memory=False)
    else:
        df = pd.read_csv(path, low_memory=False)

    cols = set(df.columns)

    # A) canonical / Cheating
    if set(FEATURE_COLS).issubset(cols):
        out = df[FEATURE_COLS].copy()
        for c in out:
            out[c] = to_num(out[c])
        return out

    # B) Tobii-style export
    tobii = {"Gaze point X", "Gaze point Y", "Pupil diameter left", "Pupil diameter right"}
    if tobii.issubset(cols):
        if "Sensor" in df.columns:
            df = df[df["Sensor"].astype(str).eq("Eye Tracker")].copy()
        if "Event" in df.columns:
            df = df[df["Event"].isna() | df["Event"].astype(str).eq("")].copy()

        out = df[["Gaze point X", "Gaze point Y", "Pupil diameter left", "Pupil diameter right"]].copy()
        for c in out:
            out[c] = to_num(out[c])

        if "Validity left" in df.columns and "Validity right" in df.columns:
            vl = df["Validity left"].astype(str)
            vr = df["Validity right"].astype(str)
            out.loc[vl.eq("Invalid"), "Pupil diameter left"] = np.nan
            out.loc[vr.eq("Invalid"), "Pupil diameter right"] = np.nan
            both_invalid = vl.eq("Invalid") & vr.eq("Invalid")
            out.loc[both_invalid, ["Gaze point X", "Gaze point Y"]] = np.nan

        return out.rename(columns={
            "Gaze point X": "Gaze X",
            "Gaze point Y": "Gaze Y",
            "Pupil diameter left": "ET_PupilLeft",
            "Pupil diameter right": "ET_PupilRight",
        })[FEATURE_COLS]

    # C) SMI / point-of-regard style
    por = {
        "rx": "Point of Regard Right X [px]",
        "ry": "Point of Regard Right Y [px]",
        "lx": "Point of Regard Left X [px]",
        "ly": "Point of Regard Left Y [px]",
        "pr": "Pupil Diameter Right [mm]",
        "pl": "Pupil Diameter Left [mm]",
    }
    if set(por.values()).issubset(cols):
        gx_r = to_num(df[por["rx"]]); gy_r = to_num(df[por["ry"]])
        gx_l = to_num(df[por["lx"]]); gy_l = to_num(df[por["ly"]])
        return pd.DataFrame({
            "Gaze X": pd.concat([gx_l, gx_r], axis=1).mean(axis=1, skipna=True),
            "Gaze Y": pd.concat([gy_l, gy_r], axis=1).mean(axis=1, skipna=True),
            "ET_PupilLeft": to_num(df[por["pl"]]),
            "ET_PupilRight": to_num(df[por["pr"]]),
        }, index=df.index)[FEATURE_COLS]

    # D) binocular raw
    d_cols = {"gaze_x_left","gaze_y_left","gaze_x_right","gaze_y_right","pupil_left","pupil_right"}
    if d_cols.issubset(cols):
        gx_l = to_num(df["gaze_x_left"]); gy_l = to_num(df["gaze_y_left"])
        gx_r = to_num(df["gaze_x_right"]); gy_r = to_num(df["gaze_y_right"])
        return pd.DataFrame({
            "Gaze X": pd.concat([gx_l, gx_r], axis=1).mean(axis=1, skipna=True),
            "Gaze Y": pd.concat([gy_l, gy_r], axis=1).mean(axis=1, skipna=True),
            "ET_PupilLeft": to_num(df["pupil_left"]),
            "ET_PupilRight": to_num(df["pupil_right"]),
        }, index=df.index)[FEATURE_COLS]

    # E) monocular raw
    if {"x", "y", "pupil"}.issubset(cols):
        out = pd.DataFrame({
            "Gaze X": to_num(df["x"]),
            "Gaze Y": to_num(df["y"]),
            "ET_PupilLeft": to_num(df["pupil"]),
            "ET_PupilRight": np.nan,
        }, index=df.index)
        if "missing" in df.columns:
            miss = pd.to_numeric(df["missing"], errors="coerce").fillna(0).astype(int).eq(1)
            out.loc[miss, ["Gaze X", "Gaze Y", "ET_PupilLeft"]] = np.nan
        return out[FEATURE_COLS]

    raise KeyError(f"Unknown format in {path.name}. First columns: {list(df.columns)[:40]}")


def sanity_check(root: Path, n=3):
    files = find_data_files(root)
    print(root, "| files:", len(files))
    for p in files[:n]:
        x = load_eye_file(p)
        print(p.name, len(x), {k: round(float(v), 3) for k, v in x.isna().mean().items()})


In [ ]:

sanity_check(D2_DIR)
sanity_check(CHEATING_DIR)


## 4. Controlled corruption models

In [ ]:

def inject_missing_mcar(df: pd.DataFrame, pct: float, seed=BASE_SEED):
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    for col in FEATURE_COLS:
        k = int(n * pct / 100.0)
        if k <= 0:
            continue
        idx = rng.choice(out.index.to_numpy(), size=k, replace=False)
        out.loc[idx, col] = np.nan
    return out


def inject_spikes(df: pd.DataFrame, pct: float, seed=BASE_SEED):
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    for col in FEATURE_COLS:
        k = int(n * pct / 100.0)
        if k <= 0:
            continue
        idx = rng.choice(out.index.to_numpy(), size=k, replace=False)
        sd = np.nanstd(out[col].to_numpy(dtype=float))
        if not np.isfinite(sd) or sd == 0:
            continue
        lam = rng.uniform(5.0, 10.0, size=k)
        out.loc[idx, col] = out.loc[idx, col].to_numpy(dtype=float) + lam * sd
    return out


## 5. Preprocessing methods

In [ ]:

def mean_impute(df):
    out = df.copy()
    active = [c for c in df.columns if df[c].notna().any()]
    if active:
        out[active] = SimpleImputer(strategy="mean").fit_transform(df[active])
    return out


def locf_impute(df):
    return df.ffill().bfill()


def knn_impute(df, k=5):
    out = df.copy()
    active = [c for c in df.columns if df[c].notna().any()]
    if active:
        out[active] = KNNImputer(n_neighbors=k).fit_transform(df[active])
    return out


def apply_imputer(df, method):
    if method == "mean":
        return mean_impute(df)
    if method == "locf":
        return locf_impute(df)
    if method == "knn(k=5)":
        return knn_impute(df, 5)
    raise ValueError(method)


def outliers_to_nan(df, method):
    out = df.copy()

    for col in out.columns:
        x = out[col].to_numpy(dtype=float)
        valid = np.isfinite(x)
        if valid.sum() < 5:
            continue

        if method == "zscore":
            zz = np.full(len(x), np.nan)
            zz[valid] = np.abs(zscore(x[valid], nan_policy="omit"))
            mask = zz > ZSCORE_THRESHOLD

        elif method == "mad":
            med = np.nanmedian(x)
            mad = median_abs_deviation(x[valid], nan_policy="omit")
            if not np.isfinite(mad) or mad == 0:
                mask = np.zeros(len(x), dtype=bool)
            else:
                mask = np.abs(x - med) > MAD_THRESHOLD * mad

        elif method == "iforest":
            mask = np.zeros(len(x), dtype=bool)
            iso = IsolationForest(
                contamination=IFOREST_CONTAMINATION,
                random_state=BASE_SEED,
                n_jobs=-1,
            )
            pred = iso.fit_predict(x[valid].reshape(-1, 1))
            mask[np.where(valid)[0][pred == -1]] = True

        else:
            raise ValueError(method)

        out.loc[out.index[mask], col] = np.nan

    return out


def apply_scaler(df, method):
    if method == "minmax":
        scaler = MinMaxScaler()
    elif method == "zscore":
        scaler = StandardScaler()
    elif method == "robust":
        scaler = RobustScaler()
    else:
        raise ValueError(method)

    out = df.copy()
    active = [c for c in df.columns if df[c].notna().any()]
    if active:
        out[active] = scaler.fit_transform(df[active])
    return out


## 6. Metrics

In [ ]:

def safe_wilcoxon(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    if len(a) < 5:
        return np.nan, np.nan
    if np.allclose(a - b, 0):
        return 0.0, 1.0
    try:
        stat, p = wilcoxon(a, b)
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan


def reconstruction_metrics(reference, corrupted, imputed, feature):
    mask = reference[feature].notna() & corrupted[feature].isna()
    y = reference.loc[mask, feature].to_numpy(dtype=float)
    yh = imputed.loc[mask, feature].to_numpy(dtype=float)
    ok = np.isfinite(y) & np.isfinite(yh)
    y, yh = y[ok], yh[ok]

    if len(y) == 0:
        return {
            "rmse": np.nan,
            "mae": np.nan,
            "wilcoxon_stat": np.nan,
            "wilcoxon_p": np.nan,
            "n_reconstructed": 0,
        }

    stat, p = safe_wilcoxon(y, yh)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y, yh))),
        "mae": float(mean_absolute_error(y, yh)),
        "wilcoxon_stat": stat,
        "wilcoxon_p": p,
        "n_reconstructed": len(y),
    }


def distribution_metrics(a, b, prefix):
    a = pd.to_numeric(a, errors="coerce").dropna().astype(float)
    b = pd.to_numeric(b, errors="coerce").dropna().astype(float)

    empty = {
        f"{prefix}_ks": np.nan,
        f"{prefix}_variance_a": np.nan,
        f"{prefix}_variance_b": np.nan,
        f"{prefix}_variance_abs_change": np.nan,
        f"{prefix}_variance_reduction_pct": np.nan,
        f"{prefix}_skew_a": np.nan,
        f"{prefix}_skew_b": np.nan,
        f"{prefix}_skew_abs_change": np.nan,
        f"{prefix}_kurt_a": np.nan,
        f"{prefix}_kurt_b": np.nan,
        f"{prefix}_kurt_abs_change": np.nan,
    }

    if len(a) < 5 or len(b) < 5:
        return empty

    va, vb = float(np.var(a)), float(np.var(b))
    sa, sb = float(skew(a)), float(skew(b))
    ka, kb = float(kurtosis(a)), float(kurtosis(b))

    return {
        f"{prefix}_ks": float(ks_2samp(a, b).statistic),
        f"{prefix}_variance_a": va,
        f"{prefix}_variance_b": vb,
        f"{prefix}_variance_abs_change": abs(vb - va),
        f"{prefix}_variance_reduction_pct": np.nan if va == 0 else (va - vb) / va * 100.0,
        f"{prefix}_skew_a": sa,
        f"{prefix}_skew_b": sb,
        f"{prefix}_skew_abs_change": abs(sb - sa),
        f"{prefix}_kurt_a": ka,
        f"{prefix}_kurt_b": kb,
        f"{prefix}_kurt_abs_change": abs(kb - ka),
    }



## 7. Complete 27-pipeline run

The paper defines two corruption families: controlled MCAR missingness and controlled spike artifacts. They are evaluated separately here so that the relevant known reference is retained for each family.

Each scenario executes all **27 complete combinations**. Intermediate stage outputs are cached within a scenario to avoid redundant computation.


In [ ]:

def evaluate_one_scenario(reference, corruption_type, level):
    if corruption_type == "missing":
        corrupted = inject_missing_mcar(reference, level, BASE_SEED)
    elif corruption_type == "outlier":
        corrupted = inject_spikes(reference, level, BASE_SEED)
    else:
        raise ValueError(corruption_type)

    rows = []

    for imp_name in IMPUTERS:
        imputed = apply_imputer(corrupted, imp_name)

        recon = {}
        for feature in FEATURE_COLS:
            if corruption_type == "missing":
                recon[feature] = reconstruction_metrics(reference, corrupted, imputed, feature)
            else:
                recon[feature] = {
                    "rmse": np.nan,
                    "mae": np.nan,
                    "wilcoxon_stat": np.nan,
                    "wilcoxon_p": np.nan,
                    "n_reconstructed": 0,
                }

        for out_name in OUTLIER_METHODS:
            cleaned = outliers_to_nan(imputed, out_name)

            out_metrics = {
                feature: distribution_metrics(reference[feature], cleaned[feature], "outlier")
                for feature in FEATURE_COLS
            }

            for scaler_name in SCALERS:
                normalized = apply_scaler(cleaned, scaler_name)

                for feature in FEATURE_COLS:
                    # Both views are retained because the paper does not fully disambiguate
                    # the upstream reference used for every normalization summary.
                    norm_local = distribution_metrics(
                        cleaned[feature], normalized[feature], "norm_local"
                    )
                    final_reference = distribution_metrics(
                        reference[feature], normalized[feature], "final_vs_reference"
                    )

                    row = {
                        "corruption_type": corruption_type,
                        "level_pct": level,
                        "imputer": imp_name,
                        "outlier_method": out_name,
                        "scaler": scaler_name,
                        "pipeline": f"{imp_name} -> {out_name} -> {scaler_name}",
                        "feature": feature,
                    }
                    row.update(recon[feature])
                    row.update(out_metrics[feature])
                    row.update(norm_local)
                    row.update(final_reference)
                    rows.append(row)

    return pd.DataFrame(rows)


In [ ]:

def file_key(path):
    return hashlib.sha1(str(path).encode("utf-8")).hexdigest()[:12]


def run_dataset(dataset_name, root, mode="full"):
    files = find_data_files(root)
    if not files:
        raise FileNotFoundError(root)

    if mode == "quick":
        files = files[:1]
        missing_levels = [5]
        outlier_levels = [5]
    else:
        missing_levels = MISSING_LEVELS_FULL
        outlier_levels = OUTLIER_LEVELS_FULL

    dataset_ckpt = CHECKPOINT_DIR / dataset_name
    dataset_ckpt.mkdir(parents=True, exist_ok=True)

    all_parts = []
    logs = []

    print(f"\n=== {dataset_name}: {len(files)} files | mode={mode} ===")

    for i, path in enumerate(files, 1):
        ckpt = dataset_ckpt / f"{file_key(path)}_{mode}.pkl"

        if USE_CHECKPOINTS and ckpt.exists() and not FORCE_RECOMPUTE:
            with open(ckpt, "rb") as f:
                part = pickle.load(f)
            all_parts.append(part)
            print(f"[{i}/{len(files)}] checkpoint {path.name}")
            continue

        print(f"[{i}/{len(files)}] run {path.name}")

        try:
            ref = load_eye_file(path).apply(pd.to_numeric, errors="coerce")
            pieces = []

            for level in missing_levels:
                pieces.append(evaluate_one_scenario(ref, "missing", level))

            for level in outlier_levels:
                pieces.append(evaluate_one_scenario(ref, "outlier", level))

            part = pd.concat(pieces, ignore_index=True)
            part.insert(0, "dataset", dataset_name)
            part.insert(1, "file", path.name)

            if USE_CHECKPOINTS:
                with open(ckpt, "wb") as f:
                    pickle.dump(part, f, protocol=pickle.HIGHEST_PROTOCOL)

            all_parts.append(part)
            logs.append({
                "dataset": dataset_name,
                "file": path.name,
                "status": "ok",
                "rows": len(ref),
                "note": "",
            })

        except Exception as e:
            logs.append({
                "dataset": dataset_name,
                "file": path.name,
                "status": "error",
                "rows": np.nan,
                "note": repr(e),
            })
            print("ERROR:", repr(e))

    raw = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
    return raw, pd.DataFrame(logs)


In [ ]:

d2_raw, d2_log = run_dataset("D2", D2_DIR, RUN_MODE)
cheating_raw, cheating_log = run_dataset("Cheating", CHEATING_DIR, RUN_MODE)

pipeline_raw = pd.concat([d2_raw, cheating_raw], ignore_index=True)
run_log = pd.concat([d2_log, cheating_log], ignore_index=True)

print("Metric rows:", len(pipeline_raw))
print("Pipelines per dataset:")
display(pipeline_raw.groupby("dataset")["pipeline"].nunique())
display(run_log.tail())


## 8. Stage-level summaries

In [ ]:

# Imputation: controlled missingness only.
stage_imputation = (
    pipeline_raw[pipeline_raw["corruption_type"].eq("missing")]
    .groupby(["dataset", "level_pct", "imputer"], as_index=False)[["rmse", "mae", "wilcoxon_p"]]
    .mean(numeric_only=True)
)

stage_imputation_global = (
    stage_imputation
    .groupby(["dataset", "imputer"], as_index=False)[["rmse", "mae", "wilcoxon_p"]]
    .mean(numeric_only=True)
)

# Outlier handling: controlled spike corruption only.
stage_outlier = (
    pipeline_raw[pipeline_raw["corruption_type"].eq("outlier")]
    .groupby(["dataset", "level_pct", "outlier_method"], as_index=False)[
        ["outlier_ks", "outlier_variance_reduction_pct"]
    ]
    .mean(numeric_only=True)
)

stage_outlier_global = (
    stage_outlier
    .groupby(["dataset", "outlier_method"], as_index=False)[
        ["outlier_ks", "outlier_variance_reduction_pct"]
    ]
    .mean(numeric_only=True)
)

# Normalization is reported in two transparent views.
stage_normalization = (
    pipeline_raw
    .groupby(["dataset", "scaler"], as_index=False)[[
        "norm_local_ks",
        "norm_local_variance_abs_change",
        "norm_local_skew_abs_change",
        "norm_local_kurt_abs_change",
        "final_vs_reference_ks",
        "final_vs_reference_variance_abs_change",
        "final_vs_reference_skew_abs_change",
        "final_vs_reference_kurt_abs_change",
    ]]
    .mean(numeric_only=True)
)

display(stage_imputation_global)
display(stage_outlier_global)
display(stage_normalization)


## 9. Pipeline-level metric vectors

In [ ]:

keys = ["dataset", "pipeline", "imputer", "outlier_method", "scaler"]

base = pipeline_raw[keys].drop_duplicates()

missing_block = (
    pipeline_raw[pipeline_raw["corruption_type"].eq("missing")]
    .groupby(keys, as_index=False)[["rmse", "mae", "wilcoxon_p"]]
    .mean(numeric_only=True)
    .rename(columns={
        "rmse": "missing_rmse",
        "mae": "missing_mae",
        "wilcoxon_p": "missing_wilcoxon_p",
    })
)

outlier_block = (
    pipeline_raw[pipeline_raw["corruption_type"].eq("outlier")]
    .groupby(keys, as_index=False)[["outlier_ks", "outlier_variance_reduction_pct"]]
    .mean(numeric_only=True)
    .rename(columns={
        "outlier_ks": "spike_outlier_ks",
        "outlier_variance_reduction_pct": "spike_outlier_variance_reduction_pct",
    })
)

shape_block = (
    pipeline_raw
    .groupby(keys, as_index=False)[[
        "norm_local_ks",
        "norm_local_variance_abs_change",
        "norm_local_skew_abs_change",
        "norm_local_kurt_abs_change",
        "final_vs_reference_ks",
        "final_vs_reference_variance_abs_change",
        "final_vs_reference_skew_abs_change",
        "final_vs_reference_kurt_abs_change",
    ]]
    .mean(numeric_only=True)
)

pipeline_joint_metrics = (
    base
    .merge(missing_block, on=keys, how="left")
    .merge(outlier_block, on=keys, how="left")
    .merge(shape_block, on=keys, how="left")
    .sort_values(["dataset", "pipeline"])
    .reset_index(drop=True)
)

assert pipeline_joint_metrics.groupby("dataset")["pipeline"].nunique().min() == 27

display(pipeline_joint_metrics.head())



## 10. Metric-specific ranks

The paper does not define one scalar overall score. Therefore ranks are reported per metric.

For RMSE, MAE, KS, and absolute shape changes, **lower is better**.


In [ ]:

rank_metrics = [
    "missing_rmse",
    "missing_mae",
    "spike_outlier_ks",
    "norm_local_ks",
    "norm_local_variance_abs_change",
    "norm_local_skew_abs_change",
    "norm_local_kurt_abs_change",
    "final_vs_reference_ks",
    "final_vs_reference_variance_abs_change",
    "final_vs_reference_skew_abs_change",
    "final_vs_reference_kurt_abs_change",
]

metric_ranks = pipeline_joint_metrics.copy()
for metric in rank_metrics:
    metric_ranks[f"{metric}_rank"] = (
        metric_ranks.groupby("dataset")[metric]
        .rank(method="min", ascending=True)
    )

REPORTED_PIPELINE = "knn(k=5) -> iforest -> minmax"
reported_pipeline = pipeline_joint_metrics[
    pipeline_joint_metrics["pipeline"].eq(REPORTED_PIPELINE)
].copy()

display(reported_pipeline)
display(
    metric_ranks[
        ["dataset", "pipeline"] + [f"{m}_rank" for m in rank_metrics]
    ].sort_values(["dataset", "pipeline"])
)


## 11. Compare recomputation with published Table 2

In [ ]:

# A compact comparison table for the stage-level quantities that are unambiguous
# in both the paper and the recomputation.

paper_imp = paper_table2[paper_table2["Stage"].eq("Imputation")].copy()
paper_out = paper_table2[paper_table2["Stage"].eq("Outlier handling")].copy()

name_map_imp = {"Mean": "mean", "LOCF": "locf", "KNN (k=5)": "knn(k=5)"}
name_map_out = {"Z-score": "zscore", "MAD": "mad", "Isolation Forest": "iforest"}

comparison_rows = []

for _, row in paper_imp.iterrows():
    method = name_map_imp[row["Method"]]
    for dataset, prefix in [("Cheating", "Cheating"), ("D2", "D2")]:
        comp = stage_imputation_global[
            (stage_imputation_global["dataset"] == dataset) &
            (stage_imputation_global["imputer"] == method)
        ]
        if len(comp):
            comparison_rows.append({
                "stage": "Imputation",
                "method": row["Method"],
                "dataset": dataset,
                "paper_rmse": row[f"{prefix}_Value1"],
                "computed_rmse": float(comp.iloc[0]["rmse"]),
                "paper_mae": row[f"{prefix}_Value2"],
                "computed_mae": float(comp.iloc[0]["mae"]),
            })

for _, row in paper_out.iterrows():
    method = name_map_out[row["Method"]]
    for dataset, prefix in [("Cheating", "Cheating"), ("D2", "D2")]:
        comp = stage_outlier_global[
            (stage_outlier_global["dataset"] == dataset) &
            (stage_outlier_global["outlier_method"] == method)
        ]
        if len(comp):
            comparison_rows.append({
                "stage": "Outlier handling",
                "method": row["Method"],
                "dataset": dataset,
                "paper_ks": row[f"{prefix}_Value1"],
                "computed_ks": float(comp.iloc[0]["outlier_ks"]),
            })

paper_vs_recomputed = pd.DataFrame(comparison_rows)
display(paper_vs_recomputed)


## 12. Export tables

In [ ]:

tag = RUN_MODE

run_log.to_csv(TABLE_DIR / f"run_log_{tag}.csv", index=False)
stage_imputation.to_csv(TABLE_DIR / f"stage_imputation_by_level_{tag}.csv", index=False)
stage_imputation_global.to_csv(TABLE_DIR / f"stage_imputation_global_{tag}.csv", index=False)
stage_outlier.to_csv(TABLE_DIR / f"stage_outlier_by_level_{tag}.csv", index=False)
stage_outlier_global.to_csv(TABLE_DIR / f"stage_outlier_global_{tag}.csv", index=False)
stage_normalization.to_csv(TABLE_DIR / f"stage_normalization_{tag}.csv", index=False)
pipeline_joint_metrics.to_csv(TABLE_DIR / f"pipeline_joint_metrics_{tag}.csv", index=False)
metric_ranks.to_csv(TABLE_DIR / f"pipeline_metric_ranks_{tag}.csv", index=False)
reported_pipeline.to_csv(TABLE_DIR / f"reported_pipeline_{tag}.csv", index=False)
paper_vs_recomputed.to_csv(TABLE_DIR / f"paper_vs_recomputed_{tag}.csv", index=False)

# Raw rows are compressed because they can become large.
pipeline_raw.to_csv(
    TABLE_DIR / f"pipeline_raw_{tag}.csv.gz",
    index=False,
    compression="gzip"
)

print("Tables written to", TABLE_DIR)


## 13. Figures

In [ ]:

# Figure 1: pipeline design space
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")

x_positions = [0.15, 0.50, 0.85]
titles = ["Imputation", "Outlier handling", "Normalization"]
methods = [
    ["Mean", "LOCF", "KNN (k=5)"],
    ["Z-score", "MAD", "Isolation Forest"],
    ["Min-Max", "Z-score", "RobustScaler"],
]

for x, title, values in zip(x_positions, titles, methods):
    ax.text(x, 0.85, title, ha="center", va="center", fontsize=14, fontweight="bold")
    for j, value in enumerate(values):
        ax.text(
            x, 0.63 - j * 0.18, value,
            ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.4", fill=False)
        )

ax.annotate("", xy=(0.42, 0.50), xytext=(0.28, 0.50), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(0.77, 0.50), xytext=(0.63, 0.50), arrowprops=dict(arrowstyle="->"))
ax.set_title("CUMULUS complete pipeline design space: 3 × 3 × 3 = 27", fontsize=15)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "pipeline_design_space.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# Figure 2: imputation RMSE by corruption level.
for dataset in stage_imputation["dataset"].dropna().unique():
    x = stage_imputation[stage_imputation["dataset"].eq(dataset)]
    fig, ax = plt.subplots(figsize=(8, 5))
    for method in IMPUTERS:
        y = x[x["imputer"].eq(method)].sort_values("level_pct")
        ax.plot(y["level_pct"], y["rmse"], marker="o", label=method)
    ax.set_xlabel("Injected missingness (%)")
    ax.set_ylabel("RMSE")
    ax.set_title(f"Imputation reconstruction error — {dataset}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"imputation_rmse_{dataset.lower()}.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:

# Figure 3: outlier KS by contamination level.
for dataset in stage_outlier["dataset"].dropna().unique():
    x = stage_outlier[stage_outlier["dataset"].eq(dataset)]
    fig, ax = plt.subplots(figsize=(8, 5))
    for method in OUTLIER_METHODS:
        y = x[x["outlier_method"].eq(method)].sort_values("level_pct")
        ax.plot(y["level_pct"], y["outlier_ks"], marker="o", label=method)
    ax.set_xlabel("Injected spike contamination (%)")
    ax.set_ylabel("KS statistic")
    ax.set_title(f"Outlier handling distributional deviation — {dataset}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"outlier_ks_{dataset.lower()}.png", dpi=200, bbox_inches="tight")
    plt.show()
